## 1. Setup and configuration


In [0]:
#Read the raw bank marketing data
bronze_df = spark.table("workspace.default.bank_marketing_bronze")

#Display a sample records
display(bronze_df)

# Inspect the column names and data types
bronze_df.printSchema()

# Verify the total number of records
print("Total rows:", bronze_df.count())

## 2. Data profiling and investigation

### Investigate Missing Values
Identify and quantify NULL values across all columns to assess 
data completeness before applying any cleaning or transformation.

In [0]:
#Handle missing values
from pyspark.sql import functions as F

#Counts NULL values in each column of the given dataframe
null_counts = bronze_df.select([F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c)
                                for c in bronze_df.columns]
                               )
#Convert the one row results into a readable verticle table
null_counts = null_counts.toPandas().transpose()

#rename the column name
null_counts.columns = ["null_count"]
                       
#Display the null counts
display(null_counts)

%md
**Finding:** The dataset does not contain any NULL values across 
all 21 columns. 

However, missing information may be represented 
using placeholder values such as "unknown".

**Decision:** No NULL value treatment is required for the current 
dataset. 

Further investigation is needed to identify and assess 
placeholder values in categorical columns.

###  Unknown Value Investigation
Identify "unknown" values across categorical columns 
to assess missing information that is not explicitly recorded as NULL.

In [0]:
# Check for "Unknown" values
# Select categorical columns
string_columns = [
    c for c, dtype in bronze_df.dtypes
    if dtype == "string"
]

# Count occurrences of "unknown" in each categorical column
unknown_counts = bronze_df.select([
    F.sum(
        (F.lower(F.col(f"`{c}`")) == "unknown").cast("int")
    ).alias(c)
    for c in string_columns
])

# Display the results vertically for easier reading
display(unknown_counts.toPandas().transpose())

In [0]:
# Convert unknown counts into a Pandas DataFrame
unknown_summary = unknown_counts.toPandas().transpose()

# Give the count column a descriptive name
unknown_summary.columns = ["unknown_count"]

# Move the column names from the index into a regular column
unknown_summary = unknown_summary.reset_index()

# Rename the index column
unknown_summary = unknown_summary.rename(columns={"index": "column_name"})

# Display column names alongside their unknown counts
display(unknown_summary)

%md
**Findings:** Six categorical columns contain "unknown" values:

- Job: 330
- Marital: 80
- Education: 1,731
- Default: 8,597
- Housing: 990
- Loan: 990

The default column has the highest number of unknown values,
accounting for approximately 21% of the dataset.

Housing and loan have identical unknown counts, suggesting a
potential relationship in how missing information was recorded.

**Decision:** Retain "unknown" as a separate category to preserve 
information and avoid introducing bias through unsupported 
assumptions or unnecessary record removal.

In [0]:
%sql
SELECT `default`, COUNT(*) AS record_count
FROM workspace.default.bank_marketing_bronze
GROUP BY `default`
ORDER BY record_count DESC;

### Investigating and identifying duplicate records

In [0]:
# Count all records in the dataset
total_rows = bronze_df.count()

# Count unique records across all columns
distinct_rows = bronze_df.distinct().count()

# Calculate the number of duplicate records
duplicate_rows = total_rows - distinct_rows

print("Total records:", total_rows)
print("Distinct records:", distinct_rows)
print("Duplicate records:", duplicate_rows)

Findings: The dataset contains 12 additional duplicate copies with each. 

In [0]:
import pyspark.sql.functions as F
# Group records by all columns and count their occurrences
duplicates = (
    bronze_df
    .groupBy(*[F.col(f"`{c}`") for c in bronze_df.columns])
    .count()
    .filter(F.col("count") > 1)  # Keep repeated records
)

In [0]:
# Display the repeated records
display(duplicates)

%md
#### Duplicate Record Investigation

- Total records: 41,188
- Distinct records: 41,176
- Additional duplicate copies: 12
- Each duplicated record appears twice.

**Decision:** Retain these records because the dataset
does not contain a unique customer or contact identifier.
We cannot establish whether they represent accidental
duplication or legitimate separate contacts.

### Numerical Data Validation

Inspect numerical column ranges and summary statistics to 
identify invalid values, unexpected observations, and 
potential data quality issues before applying transformations.

In [0]:
# Generate summary statistics for selected numerical columns
numeric_summary = bronze_df.select(
    "age",
    "duration",
    "campaign",
    "pdays",
    "previous"
).describe()

# Display summary statistics
display(numeric_summary)


%md
### Economic Indicator Investigation

Examine the summary statistics and numerical ranges of 
economic indicators to identify potential data quality issues.

In [0]:
# Select economic indicators and temporarily alias column names
economic_df = bronze_df.select(
    F.col("`emp.var.rate`").alias("emp_var_rate"),
    F.col("`cons.price.idx`").alias("cons_price_idx"),
    F.col("`cons.conf.idx`").alias("cons_conf_idx"),
    F.col("euribor3m"),
    F.col("`nr.employed`").alias("nr_employed")
)

# Generate and display summary statistics
display(economic_df.describe())

%md
### Categorical Data Validation

Inspect distinct values across categorical columns to identify
inconsistent categories, unexpected values, and potential
data quality issues.

In [0]:
# Identify categorical columns based on their data type
categorical_columns = [
    c for c, dtype in bronze_df.dtypes
    if dtype == "string"
]

# Retrieve and display distinct values for each categorical column
for c in categorical_columns:
    values = bronze_df.select(c).distinct().collect()
    print(f"{c}: {[row[0] for row in values]}")

### Empty string investigation

In [0]:
# Check each categorical column for empty or whitespace-only values
for c in categorical_columns:

    # Remove surrounding spaces and count empty strings
    empty_count = bronze_df.filter(F.trim(F.col(c)) == "").count()

    # Display the column name and empty string count
    print(f"{c}: {empty_count}")

## 3. Data Cleaning and Transformation
Apply standardized cleaning and transformation rules based 
on the findings from the data profiling stage. These rules 
will ensure consistent processing of future campaign data 
while preserving the integrity of the original records.

### Standardization

In [0]:
# Replace periods in column names with underscores
silver_df = bronze_df.toDF(
    *[c.replace(".", "_") for c in bronze_df.columns]
)

# Rename default to avoid SQL keyword conflicts
silver_df = silver_df.withColumnRenamed("default", "credit_default")

# Verify the standardized schema
silver_df.printSchema()

#### Categorical Value Standardization

Standardize job and education category labels using underscores
and remove unnecessary punctuation to improve consistency
and readability while preserving their original meanings.

In [0]:
# Replace periods in education values with underscores
silver_df = silver_df.withColumn(
    "education",
    F.regexp_replace(F.col("education"), r"\.", "_")
)

# Replace hyphens in job values with underscores
silver_df = silver_df.withColumn(
    "job",
    F.regexp_replace(F.col("job"), "-", "_")
)

# Remove trailing periods from job values
silver_df = silver_df.withColumn(
    "job",
    F.regexp_replace(F.col("job"), r"\.$", "")
)

### Special Value Handling

The pdays column contains 999 as a placeholder indicating
that a customer was not contacted during a previous campaign.

I am creating a binary indicator to preserve this information and
replace the placeholder with NULL to accurately represent
the absence of elapsed contact days.

In [0]:
# Indicate whether a customer was previously contacted
silver_df = silver_df.withColumn(
    "previously_contacted",
    F.when(F.col("pdays") == 999, 0).otherwise(1)
)

In [0]:
# Replace the sentinel value with NULL
silver_df = silver_df.withColumn(
    "pdays",
    F.when(F.col("pdays") == 999, None)
    .otherwise(F.col("pdays"))
)

In [0]:
# Inspect the current schema
silver_df.printSchema()

## 4. Data Quality Vaildation


### Schema Validation

Verify that all required columns exist and have the
expected data types.

In [0]:
actual_schema = dict(silver_df.dtypes)
print(actual_schema)

In [0]:
# Define the expected Silver schema
expected_schema = {
    "age": "int",
    "job": "string",
    "marital": "string",
    "education": "string",
    "credit_default": "string",
    "housing": "string",
    "loan": "string",
    "contact": "string",
    "month": "string",
    "day_of_week": "string",
    "duration": "int",
    "campaign": "int",
    "pdays": "int",
    "previous": "int",
    "poutcome": "string",
    "emp_var_rate": "double",
    "cons_price_idx": "double",
    "cons_conf_idx": "double",
    "euribor3m": "double",
    "nr_employed": "double",
    "y": "string",
    "previously_contacted": "int"
}

In [0]:
# Validate the Silver schema
if actual_schema != expected_schema:
    raise ValueError("Silver schema validation failed.")
else:
    print("Silver schema validation passed.")

#### Completeness Validation

In [0]:
# Identify columns that must not contain NULL values
required_columns = [
    c for c in expected_schema if c != "pdays"
]
print(required_columns)

In [0]:

# Validate completeness of required columns
for c in required_columns:
    null_count = silver_df.filter(F.col(c).isNull()).count()

    if null_count > 0:
        raise ValueError(f"{c} contains {null_count} NULL values.")

print("Completeness validation passed.")

### Data Validation

In [0]:
invalid_age_count = silver_df.filter(F.col("age") <= 0).count()

if invalid_age_count > 0:
    raise ValueError("Invalid age values detected.")

In [0]:
# Define minimum acceptable values
numerical_rules = {
    "age": 1,
    "duration": 0,
    "campaign": 1,
    "pdays": 0,
    "previous": 0
}

In [0]:
# Validate numerical values against minimum requirements
for c, min_value in numerical_rules.items():
    invalid_count = silver_df.filter(
        F.col(c) < min_value
    ).count()

    if invalid_count > 0:
        raise ValueError(f"{c} contains {invalid_count} invalid values.")

print("Numerical validity validation passed.")

In [0]:
# Count records with an invalid subscription outcome
invalid_y_count = silver_df.filter(
    ~F.col("y").isin("yes", "no")
).count()

print("Invalid target values:", invalid_y_count)

In [0]:
# Stop the pipeline if invalid target values exist
if invalid_y_count > 0:
    raise ValueError(
        f"Target column contains {invalid_y_count} invalid values."
    )

print("Target validity validation passed.")

In [0]:
categorical_rules = {
    "y": ["yes", "no"],
    "housing": ["yes", "no", "unknown"],
    "loan": ["yes", "no", "unknown"],
    "credit_default": ["yes", "no", "unknown"],
    "contact": ["cellular", "telephone"]
}

In [0]:
# Check categorical columns against their accepted values
for c, allowed_values in categorical_rules.items():

    invalid_count = silver_df.filter(
        ~F.col(c).isin(allowed_values)
    ).count()

    if invalid_count > 0:
        raise ValueError(
            f"{c} contains {invalid_count} invalid values."
        )

print("Categorical validity validation passed.")

In [0]:
categorical_rules.update({
    "marital": ["divorced", "married", "single", "unknown"],
    "education": [
        "basic_4y", "basic_6y", "basic_9y",
        "high_school", "illiterate", "professional_course",
        "university_degree", "unknown"
    ],
    "job": [
        "admin", "blue_collar", "entrepreneur", "housemaid",
        "management", "retired", "self_employed", "services",
        "student", "technician", "unemployed", "unknown"
    ],
    "month": [
        "jan", "feb", "mar", "apr", "may", "jun",
        "jul", "aug", "sep", "oct", "nov", "dec"
    ],
    "day_of_week": ["mon", "tue", "wed", "thu", "fri"],
    "poutcome": ["failure", "nonexistent", "success"]
})

In [0]:
for c, allowed_values in categorical_rules.items():
    invalid_count = silver_df.filter(
        ~F.col(c).isin(allowed_values)
    ).count()

    if invalid_count > 0:
        raise ValueError(
            f"{c} contains {invalid_count} invalid values."
        )

print("All categorical validity checks passed.")

### Consistency Validation

In [0]:
# Identify records where the two columns disagree
invalid_records = silver_df.filter(
    ((F.col("previously_contacted") == 0) & F.col("pdays").isNotNull()) |
    ((F.col("previously_contacted") == 1) & F.col("pdays").isNull())
)

# Count inconsistent records
invalid_count = invalid_records.count()

if invalid_count > 0:
    raise ValueError(
        f"Found {invalid_count} inconsistent records."
    )

# Ensure the indicator contains only 0 or 1
invalid_indicator_count = silver_df.filter(
    F.col("previously_contacted").isNull() |
    ~F.col("previously_contacted").isin(0, 1)
).count()

if invalid_indicator_count > 0:
    raise ValueError("Invalid previously_contacted values detected.")

print("Consistency validation passed.")

### Record Preservation

In [0]:
# Ensure no records were lost or added during transformation
bronze_count = bronze_df.count()
silver_count = silver_df.count()

if bronze_count != silver_count:
    raise ValueError(
        f"Record count mismatch: Bronze={bronze_count}, Silver={silver_count}"
    )

print("Record preservation validation passed.")

In [0]:
# Save validated records
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bank_marketing_silver")
)

print("Silver Delta table written successfully.")

In [0]:
# Verify the saved Silver table
saved_silver_df = spark.table("workspace.default.bank_marketing_silver")

print("Silver table records:", saved_silver_df.count())
display(saved_silver_df.limit(10))

# 